# 1. Modeling Objective

This notebook trains a leakage-safe CatBoost binary classifier that estimates `P(label = 1 | claim features)`. It ranks claims for a limited audit portfolio; it does not make autonomous fraud decisions or optimize a fixed 0.5 threshold.

## 2. Imports and Configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import platform

import catboost
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from catboost import Pool
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from prs_its.calibration import (
    calibrate_test_predictions,
    calibration_comparison,
    calibration_curve_frame,
    cross_fit_calibration,
    prediction_distribution,
    should_select_calibration,
)
from prs_its.fairness import age_groups, fairness_across_budgets
from prs_its.metrics import audit_metrics, evaluate_probabilities
from prs_its.modeling import (
    BASE_PARAMS,
    ID_COL,
    N_SPLITS,
    RANDOM_STATE,
    TARGET,
    aggregate_feature_importance,
    code_like_dtypes,
    ensure_gpu_ready,
    make_feature_spec,
    prepare_catboost_features,
    train_catboost_cv,
    validate_train_test_schema,
)
from prs_its.submission import make_submission, prediction_summary

TASK_TYPE = os.environ.get('PRS_ITS_TASK_TYPE', 'GPU').upper()
GPU_DEVICES = os.environ.get('PRS_ITS_GPU_DEVICES', '0')
EARLY_STOPPING_ROUNDS = 200

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate pyproject.toml.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = OUTPUT_DIR / 'models'
OOF_DIR = OUTPUT_DIR / 'oof'
METRICS_DIR = OUTPUT_DIR / 'metrics'
FIGURES_DIR = OUTPUT_DIR / 'figures'
SUBMISSIONS_DIR = OUTPUT_DIR / 'submissions'
for directory in [MODELS_DIR, OOF_DIR, METRICS_DIR, FIGURES_DIR, SUBMISSIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

versions = pd.Series({
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'catboost': catboost.__version__,
    'scikit_learn': sklearn.__version__,
    'task_type': TASK_TYPE,
    'gpu_devices': GPU_DEVICES,
})
display(versions)

## 3. Load Data

In [ ]:
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
train = pd.read_csv(TRAIN_PATH, dtype=code_like_dtypes())
test = pd.read_csv(TEST_PATH, dtype=code_like_dtypes())

assert ID_COL in train.columns
assert ID_COL in test.columns
assert TARGET in train.columns
assert TARGET not in test.columns
assert set(train[TARGET].dropna().unique()).issubset({0, 1})

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Fraud prevalence: {train[TARGET].mean():.2%}')

## 4. Schema and Leakage Validation

In [ ]:
FEATURES = validate_train_test_schema(train, test)
EXCLUDED_FEATURES = [ID_COL]
assert ID_COL not in FEATURES
assert TARGET not in FEATURES
assert set(FEATURES) == set(test.columns) - {ID_COL}

schema_rows = []
for column in FEATURES:
    row = {'feature': column, 'train_dtype': str(train[column].dtype), 'test_dtype': str(test[column].dtype)}
    if column in code_like_dtypes():
        train_values = set(train[column].astype('string').dropna())
        test_values = test[column].astype('string')
        row['test_unseen_categories'] = int((~test_values.isin(train_values) & test_values.notna()).sum())
        row['test_unseen_pct'] = row['test_unseen_categories'] / len(test) * 100
    else:
        row['test_unseen_categories'] = np.nan
        row['test_unseen_pct'] = np.nan
    row['train_missing_pct'] = train[column].isna().mean() * 100
    row['test_missing_pct'] = test[column].isna().mean() * 100
    schema_rows.append(row)
schema_check = pd.DataFrame(schema_rows)
numeric_distribution_check = pd.DataFrame([
    {'feature': column, 'train_min': pd.to_numeric(train[column], errors='coerce').min(), 'train_max': pd.to_numeric(train[column], errors='coerce').max(), 'test_min': pd.to_numeric(test[column], errors='coerce').min(), 'test_max': pd.to_numeric(test[column], errors='coerce').max()}
    for column in ['umur', 'los'] if column in FEATURES
])
display(schema_check)
display(numeric_distribution_check)
print('claim_id is retained only for integrity checks, OOF records, and submission generation.')

## 5. Feature Definitions

In [ ]:
FEATURE_SPEC = make_feature_spec(train, test)
CATEGORICAL_FEATURES = FEATURE_SPEC.categorical_features
NUMERIC_FEATURES = FEATURE_SPEC.numeric_features
BINARY_FEATURES = FEATURE_SPEC.binary_features
COUNT_FEATURES = FEATURE_SPEC.count_features
display(pd.Series(FEATURE_SPEC.as_dict()))
print('Diagnosis and procedure semantics remain provisional until the official data dictionary is available.')

## 6. CatBoost Data Preparation

In [ ]:
baseline_data = prepare_catboost_features(train, test, FEATURE_SPEC, add_count_features=False)
count_feature_data = prepare_catboost_features(train, test, FEATURE_SPEC, add_count_features=True)
X = baseline_data.X
y = baseline_data.y
X_test = baseline_data.X_test
assert list(X.columns) == list(X_test.columns)
display(X.dtypes.to_frame('dtype'))

## 7. Competition Metrics

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
random_pred = rng.random(len(y))
constant_pred = np.full(len(y), y.mean())
baseline_metrics = pd.DataFrame([
    {'experiment_name': 'random', **evaluate_probabilities(y, random_pred)},
    {'experiment_name': 'constant_prevalence', **evaluate_probabilities(y, constant_pred)},
])
display(baseline_metrics)

## 8. Cross-Validation Strategy

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cv_settings = {'type': 'StratifiedKFold', 'n_splits': N_SPLITS, 'shuffle': True, 'random_state': RANDOM_STATE}
display(pd.Series(cv_settings))
print('No patient, provider, temporal, or independent claim-group identifier is available. Repeated feature profiles remain a documented limitation.')

## 9. Baseline CatBoost

GPU training is required for the full experiment run. Set `PRS_ITS_TASK_TYPE=CPU` only for an intentional CPU fallback.

In [ ]:
if TASK_TYPE == 'GPU':
    GPU_STATUS = ensure_gpu_ready(GPU_DEVICES)
    print(GPU_STATUS)
else:
    GPU_STATUS = 'CPU explicitly selected'
    print(GPU_STATUS)

BASELINE_PARAMS = BASE_PARAMS.copy()
display(pd.Series(BASELINE_PARAMS))

## 10. OOF Training

In [ ]:
def run_experiment(name, prepared, params, notes):
    result = train_catboost_cv(
        prepared.X, prepared.y, prepared.X_test, CATEGORICAL_FEATURES, cv=cv, params=params,
        task_type=TASK_TYPE, devices=GPU_DEVICES, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    oof_metrics = evaluate_probabilities(prepared.y, result['oof_pred'])
    result.update({'experiment_name': name, 'notes': notes, 'oof_metrics': oof_metrics, 'prepared': prepared})
    return result

EXPERIMENT_SPECS = [
    ('unweighted_baseline', baseline_data, BASELINE_PARAMS, 'Original features; no class weighting.'),
    ('balanced_baseline', baseline_data, {**BASELINE_PARAMS, 'auto_class_weights': 'Balanced'}, 'Original features; CatBoost balanced weights.'),
    ('count_features', count_feature_data, BASELINE_PARAMS, 'Original features plus diagnosis and procedure counts.'),
    ('shallow_regularized', baseline_data, {**BASELINE_PARAMS, 'depth': 4, 'l2_leaf_reg': 10.0, 'random_strength': 0.5, 'bagging_temperature': 0.5}, 'Targeted shallow regularization.'),
    ('deep_regularized', baseline_data, {**BASELINE_PARAMS, 'depth': 8, 'l2_leaf_reg': 10.0, 'random_strength': 2.0, 'bagging_temperature': 1.0}, 'Targeted deeper regularization.'),
]
experiment_runs = {}
for name, prepared, params, notes in EXPERIMENT_SPECS:
    experiment_runs[name] = run_experiment(name, prepared, params, notes)
    print(f'Completed {name}')
    experiment_runs[name]['models'].clear()

assert all((run['fold_id'] >= 0).all() for run in experiment_runs.values())

## 11. Audit-Budget Evaluation

In [ ]:
audit_rows = []
for name, run in experiment_runs.items():
    for fraction in (0.03, 0.05, 0.07):
        audit_rows.append({'experiment_name': name, **audit_metrics(y, run['oof_pred'], fraction)})
audit_results = pd.DataFrame(audit_rows)
display(audit_results)

## 12. Ranking Evaluation

In [ ]:
experiment_rows = []
for name, run in experiment_runs.items():
    fold_metrics = run['fold_metrics']
    experiment_rows.append({
        'experiment_name': name, 'feature_set': 'count_features' if name == 'count_features' else 'original',
        'params': json.dumps(run['params'], sort_keys=True, default=str),
        'class_weight_strategy': run['params'].get('auto_class_weights', 'None'), 'cv_strategy': cv_settings['type'],
        **run['oof_metrics'], 'mean_best_iteration': fold_metrics['best_iteration'].mean(),
        'fold_normalized_recall_5_std': fold_metrics['normalized_recall_at_5pct'].std(), 'notes': run['notes'],
    })
experiment_results = pd.DataFrame(experiment_rows)
display(experiment_results.sort_values(['normalized_recall_at_5pct', 'average_precision'], ascending=False))

## 13. Calibration Evaluation

In [ ]:
def fairness_gap(probabilities):
    rates = pd.concat([
        fairness_across_budgets(train['jkpst'], y, probabilities, group_name='jkpst').query('audit_fraction == 0.05 and eligible_for_comparison'),
        fairness_across_budgets(age_groups(train['umur']), y, probabilities, group_name='age_group').query('audit_fraction == 0.05 and eligible_for_comparison'),
    ])
    if rates.empty:
        return np.nan
    return rates['audit_rate'].max() - rates['audit_rate'].min()

experiment_results['fairness_audit_rate_gap_5'] = [fairness_gap(experiment_runs[name]['oof_pred']) for name in experiment_results['experiment_name']]
best_normalized_recall = experiment_results['normalized_recall_at_5pct'].max()
selection_pool = experiment_results.query('normalized_recall_at_5pct >= @best_normalized_recall - 0.005').copy()
selected_name = selection_pool.sort_values(
    ['average_precision', 'brier_score', 'fold_normalized_recall_5_std', 'fairness_audit_rate_gap_5', 'mean_best_iteration'],
    ascending=[False, True, True, True, True],
    na_position='last',
).iloc[0]['experiment_name']
print(f'Selected raw configuration: {selected_name}')

selected = experiment_runs[selected_name]
final_run = train_catboost_cv(
    selected['prepared'].X, y, selected['prepared'].X_test, CATEGORICAL_FEATURES, cv=cv, params=selected['params'],
    task_type=TASK_TYPE, devices=GPU_DEVICES, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    model_dir=MODELS_DIR, model_prefix='catboost',
)
raw_oof_pred = final_run['oof_pred']
raw_test_pred = final_run['test_pred']
raw_metrics = evaluate_probabilities(y, raw_oof_pred)
calibration_candidates = []
for method in ('sigmoid', 'isotonic'):
    calibrated = cross_fit_calibration(y, raw_oof_pred, final_run['fold_id'], method)
    metrics = evaluate_probabilities(y, calibrated['oof_pred'])
    calibration_candidates.append((method, calibrated['oof_pred'], metrics))
calibration_table = pd.DataFrame([{'prediction_type': 'raw', **raw_metrics}] + [
    {'prediction_type': method, **metrics} for method, _, metrics in calibration_candidates
])
display(calibration_table)
eligible_calibration = [candidate for candidate in calibration_candidates if should_select_calibration(raw_metrics, candidate[2])]
if eligible_calibration:
    calibration_method, final_oof_pred, final_metrics = min(eligible_calibration, key=lambda item: item[2]['brier_score'])
    final_test_pred = calibrate_test_predictions(raw_oof_pred, y, raw_test_pred, calibration_method)
else:
    calibration_method, final_oof_pred, final_metrics, final_test_pred = 'raw', raw_oof_pred, raw_metrics, raw_test_pred
print(f'Final probability treatment: {calibration_method}')
calibration_curve_data = calibration_curve_frame(y, final_oof_pred)
display(calibration_curve_data)

## 14. Class-Imbalance Experiments

In [ ]:
weighting_comparison = experiment_results.query("experiment_name in ['unweighted_baseline', 'balanced_baseline']")
display(weighting_comparison)
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='ideal')
for name in ['unweighted_baseline', 'balanced_baseline']:
    curve = calibration_curve_frame(y, experiment_runs[name]['oof_pred'])
    plt.plot(curve['mean_predicted_probability'], curve['observed_fraud_rate'], marker='o', label=name)
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed fraud rate')
plt.legend()
plt.title('Class-Weighting Calibration Comparison')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_weighting_calibration.png', dpi=160)
plt.show()
print('Oversampling is intentionally not used. Weighting results are compared on OOF ranking and probability quality.')

## 15. Hyperparameter Experiments

In [ ]:
display(experiment_results[['experiment_name', 'average_precision', 'brier_score', 'normalized_recall_at_5pct', 'precision_at_5pct', 'lift_at_5pct', 'mean_best_iteration']])
experiment_results.to_csv(METRICS_DIR / 'catboost_experiments.csv', index=False)

## 16. Feature Importance / Explainability

In [ ]:
feature_importance = aggregate_feature_importance(final_run['feature_importance'])
display(feature_importance.head(25))
ax = feature_importance.head(25).sort_values('mean_importance').plot.barh(x='feature', y='mean_importance', legend=False, figsize=(10, 8))
ax.set_title('CatBoost Predictive Contribution Across Folds')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'catboost_feature_importance.png', dpi=160)
plt.show()

sample_size = min(2000, len(y))
sample_indices = np.random.default_rng(RANDOM_STATE).choice(len(y), size=sample_size, replace=False)
shap_rows = []
for fold, model in enumerate(final_run['models']):
    indices = sample_indices[final_run['fold_id'][sample_indices] == fold]
    if len(indices):
        values = model.get_feature_importance(Pool(selected['prepared'].X.iloc[indices], cat_features=CATEGORICAL_FEATURES), type='ShapValues')[:, :-1]
        shap_rows.append(pd.DataFrame({'feature': selected['prepared'].X.columns, 'mean_abs_shap': np.abs(values).mean(axis=0), 'fold': fold}))
shap_summary = pd.concat(shap_rows).groupby('feature', as_index=False)['mean_abs_shap'].mean().sort_values('mean_abs_shap', ascending=False)
display(shap_summary.head(25))

representative_indices = {
    'high_risk_fraud': int(np.flatnonzero((y.to_numpy() == 1) & (final_oof_pred >= np.quantile(final_oof_pred, 0.95)))[0]),
    'high_risk_legitimate': int(np.flatnonzero((y.to_numpy() == 0) & (final_oof_pred >= np.quantile(final_oof_pred[y.to_numpy() == 0], 0.95)))[0]),
    'missed_fraud': int(np.flatnonzero((y.to_numpy() == 1) & (final_oof_pred <= np.quantile(final_oof_pred[y.to_numpy() == 1], 0.05)))[0]),
    'low_risk_legitimate': int(np.flatnonzero((y.to_numpy() == 0) & (final_oof_pred <= np.quantile(final_oof_pred[y.to_numpy() == 0], 0.05)))[0]),
}
explanation_rows = []
for case, index in representative_indices.items():
    model = final_run['models'][final_run['fold_id'][index]]
    values = model.get_feature_importance(Pool(selected['prepared'].X.iloc[[index]], cat_features=CATEGORICAL_FEATURES), type='ShapValues')[0, :-1]
    for feature_index in np.argsort(np.abs(values))[-5:][::-1]:
        explanation_rows.append({'case': case, 'claim_id': train.iloc[index][ID_COL], 'label': int(y.iloc[index]), 'oof_fraud_probability': final_oof_pred[index], 'feature': selected['prepared'].X.columns[feature_index], 'shap_contribution': values[feature_index]})
representative_explanations = pd.DataFrame(explanation_rows)
display(representative_explanations)
print('Importance and SHAP values describe model behavior and predictive association, not causal evidence of fraud.')

## 17. Policyholder Protection

In [ ]:
gender_fairness = fairness_across_budgets(train['jkpst'], y, final_oof_pred, group_name='jkpst')
age_fairness = fairness_across_budgets(age_groups(train['umur']), y, final_oof_pred, group_name='age_group')
fairness_results = pd.concat([gender_fairness, age_fairness], ignore_index=True)
fairness_results.to_csv(METRICS_DIR / 'catboost_fairness.csv', index=False)
display(fairness_results.query('audit_fraction == 0.05'))
print('Rates are calculated only among legitimate claims. Raw jkpst categories are not assigned demographic meanings without the data dictionary.')

## 18. Robustness Analysis

In [ ]:
robustness = pd.DataFrame([audit_metrics(y, final_oof_pred, fraction) for fraction in np.arange(0.01, 0.101, 0.01)])
display(robustness)
ax = robustness.plot(x='audit_fraction', y='fraud_caught', marker='o', legend=False)
ax.set_title('Fraud Capture by Audit Capacity')
ax.set_xlabel('Audit fraction')
ax.set_ylabel('Fraud claims captured')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fraud_capture_curve.png', dpi=160)
plt.show()
fold_metrics = final_run['fold_metrics']
fold_metrics.to_csv(METRICS_DIR / 'catboost_fold_metrics.csv', index=False)
display(fold_metrics)
display(prediction_distribution(y, final_oof_pred))
plt.figure(figsize=(10, 5))
plt.hist(final_oof_pred[y.to_numpy() == 0], bins=40, alpha=0.6, label='label = 0')
plt.hist(final_oof_pred[y.to_numpy() == 1], bins=40, alpha=0.6, label='label = 1')
plt.legend()
plt.title('OOF Fraud-Probability Distribution by Label')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'oof_probability_distribution.png', dpi=160)
plt.show()
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='ideal')
plt.plot(calibration_curve_data['mean_predicted_probability'], calibration_curve_data['observed_fraud_rate'], marker='o', label=calibration_method)
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed fraud rate')
plt.title('OOF Reliability Curve')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'calibration_curve.png', dpi=160)
plt.show()

## 19. Final Model Selection

In [ ]:
FINAL_PARAMS = selected['params']
FINAL_CONFIG = {
    'model': 'CatBoostClassifier', 'features': list(selected['prepared'].X.columns),
    'categorical_features': CATEGORICAL_FEATURES, 'excluded_features': EXCLUDED_FEATURES,
    'params': FINAL_PARAMS, 'cv': cv_settings, 'calibration': calibration_method,
    'gpu_status': GPU_STATUS, 'random_state': RANDOM_STATE,
}
with (MODELS_DIR / 'catboost_final_config.json').open('w') as file:
    json.dump(FINAL_CONFIG, file, indent=2, default=str)
display(pd.Series({'selected_experiment': selected_name, **final_metrics}))

## 20. Test Prediction

In [ ]:
oof_output = pd.DataFrame({
    ID_COL: train[ID_COL], TARGET: y, 'fold': final_run['fold_id'],
    'fraud_probability_raw': raw_oof_pred, 'fraud_probability_final': final_oof_pred,
})
oof_output.to_csv(OOF_DIR / 'catboost_oof.csv', index=False)
assert len(oof_output) == len(train)
assert oof_output['fold'].ge(0).all()
assert oof_output[['fraud_probability_raw', 'fraud_probability_final']].notna().all().all()
display(prediction_summary(final_test_pred))

## 21. Submission Generation

In [ ]:
submission = make_submission(test[ID_COL], final_test_pred, SUBMISSIONS_DIR / 'catboost_submission.csv')
assert len(submission) == len(test)
assert submission[ID_COL].equals(test[ID_COL].reset_index(drop=True))
assert submission['fraud_probability'].notna().all()
assert np.isfinite(submission['fraud_probability']).all()
assert submission['fraud_probability'].between(0, 1).all()
display(submission.head())

## 22. Limitations

Feature semantics and decision-time availability remain unconfirmed without the official data dictionary. Repeated feature profiles, possible train/test category shift, private-test uncertainty, and lack of temporal or independent-entity validation constrain generalization claims. Probability calibration and legitimate-policyholder audit exposure must be monitored after deployment.

## 23. Final Findings

In [ ]:
final_findings = pd.DataFrame([
    {'Finding': 'Selected CatBoost configuration', 'Evidence': selected_name, 'Decision': selected_name, 'Reason': 'Selection uses OOF Normalized Recall@5%, AP, Brier, stability, fairness gap, and complexity.'},
    {'Finding': 'Final probability treatment', 'Evidence': calibration_table.to_json(orient='records'), 'Decision': calibration_method, 'Reason': 'Calibration is selected only when Brier improves without unacceptable ranking loss.'},
    {'Finding': 'Audit portfolio at 5%', 'Evidence': json.dumps(audit_metrics(y, final_oof_pred, 0.05)), 'Decision': 'Human audit prioritization', 'Reason': 'The model ranks claims; investigators determine fraud.'},
])
display(final_findings)
final_findings.to_csv(METRICS_DIR / 'catboost_final_findings.csv', index=False)
print('All conclusions are based on OOF predictions and do not establish causal relationships.')